In [1]:
# Load Libraries
import pandas as pd
import numpy as np

In [4]:
# Load Data
DATA_PATH = "../data/raw/CommunityHealthcare_MultiSymptomsDisease_Diagnostic_Dataset.csv"

In [5]:
# Read Data
df = pd.read_csv(DATA_PATH)

In [6]:
# Define Target 
TARGET = "disease"

In [7]:
# Validate the Dataset
if TARGET not in df.columns:
    raise ValueError(f"Missing target column: {TARGET}")

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Number of disease classes:", df[TARGET].nunique())

Dataset shape: (100000, 175)
Missing values: 0
Duplicate rows: 0
Number of disease classes: 46


In [8]:
# Validate Symptom Columns
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print("Input features:", X.shape[1])
print("Target classes:", y.nunique())

print("\nData types:")
print(X.dtypes.value_counts())

print("\nUnique feature values:")
for value in sorted(X.stack().unique()):
    print(value)

Input features: 174
Target classes: 46

Data types:
int64    174
Name: count, dtype: int64

Unique feature values:
0
1


In [9]:
invalid_values = {
    col: sorted(X[col].dropna().unique().tolist())
    for col in X.columns
    if not X[col].dropna().isin([0, 1]).all()
}

if invalid_values:
    print("Features with unexpected values:")
    print(invalid_values)
else:
    print("All symptom features contain only 0 and 1.")

All symptom features contain only 0 and 1.


In [ ]:
# Check Class Imbalance
class_counts = y.value_counts()

print("Smallest class sizes:")
print(class_counts.sort_values().head(15))

print("\nLargest class sizes:")
print(class_counts.sort_values(ascending=False).head(10))

print("\nClasses with fewer than 10 records:",
      (class_counts < 10).sum())

print("Classes with fewer than 2 records:",
      (class_counts < 2).sum())

Total disease classes: 46
Classes with fewer than 10 records: 9
Classes with only one record: 1

Rare disease classes:
disease
Metabolic_Syndrome         1
Urinary_Tract_Infection    2
Typhoid_Fever              2
Psoriasis                  2
Acute_Gastroenteritis      3
Crohn_Disease              4
Fanconi_Anemia             4
Chronic_Liver_Disease      5
Ulcerative_Colitis         7
Name: count, dtype: int64


In [25]:
# Count records for each disease
class_counts = df["disease"].value_counts().sort_values()

# Display diseases with fewer than 10 records
rare_classes = class_counts[class_counts < 10]

print("Total disease classes:", df["disease"].nunique())
print("Classes with fewer than 10 records:", len(rare_classes))
print("Classes with only one record:", (class_counts == 1).sum())

print("\nRare disease classes:")
print(rare_classes)

Total disease classes: 46
Classes with fewer than 10 records: 9
Classes with only one record: 1

Rare disease classes:
disease
Metabolic_Syndrome         1
Urinary_Tract_Infection    2
Typhoid_Fever              2
Psoriasis                  2
Acute_Gastroenteritis      3
Crohn_Disease              4
Fanconi_Anemia             4
Chronic_Liver_Disease      5
Ulcerative_Colitis         7
Name: count, dtype: int64


In [26]:
# Encoding Disease Variables 
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Number of encoded classes:",
      len(label_encoder.classes_))

for index, disease in enumerate(label_encoder.classes_):
    print(index, "->", disease)

Number of encoded classes: 46
0 -> Acute_Gastroenteritis
1 -> Amyotrophic_Lateral_Sclerosis
2 -> Asthma
3 -> COPD
4 -> COVID19
5 -> Chronic_Kidney_Disease
6 -> Chronic_Liver_Disease
7 -> Cirrhosis
8 -> Common_Cold
9 -> Coronary_Artery_Disease
10 -> Crohn_Disease
11 -> Cystic_Fibrosis
12 -> Dengue
13 -> Depression
14 -> Ehlers_Danlos_Syndrome
15 -> Epilepsy
16 -> Fanconi_Anemia
17 -> Flu
18 -> Gastritis
19 -> Gaucher_Disease
20 -> Generalized_Anxiety_Disorder
21 -> Hypertension
22 -> Hyperthyroidism
23 -> Hypothyroidism
24 -> Inflammatory_Bowel_Disease
25 -> Irritable_Bowel_Syndrome
26 -> Marfan_Syndrome
27 -> Metabolic_Syndrome
28 -> Migraine
29 -> Multiple_Sclerosis
30 -> Niemann_Pick_Disease
31 -> Obesity
32 -> Osteoarthritis
33 -> Osteopetrosis
34 -> Peptic_Ulcer_Disease
35 -> Pneumonia
36 -> Polycystic_Ovary_Syndrome
37 -> Psoriasis
38 -> Spinal_Muscular_Atrophy
39 -> Stroke
40 -> Systemic_Lupus_Erythematosus
41 -> Tuberculosis
42 -> Type2_Diabetes
43 -> Typhoid_Fever
44 -> Ulcerat

In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder



In [29]:
# Separate input features and target
X = df.drop(columns=["disease"]).copy()
y = df["disease"].copy()

# Encode disease names
label_encoder = LabelEncoder()
y_encoded = pd.Series(
    label_encoder.fit_transform(y),
    index=y.index,
    name="disease_encoded"
)

# Identify classes with one record
class_counts = y.value_counts()
singleton_classes = class_counts[class_counts < 2].index

# Keep singleton records in training
singleton_mask = y.isin(singleton_classes)

X_single = X.loc[singleton_mask]
y_single = y_encoded.loc[singleton_mask]

# Split the remaining records using stratification
X_remaining = X.loc[~singleton_mask]
y_remaining = y_encoded.loc[~singleton_mask]

X_train_rest, X_test, y_train_rest, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.20,
    random_state=42,
    stratify=y_remaining
)

# Combine singleton records with training data
X_train = pd.concat([X_train_rest, X_single])
y_train = pd.concat([y_train_rest, y_single])

# Shuffle training records
train_order = np.random.RandomState(42).permutation(len(X_train))

X_train = X_train.iloc[train_order].reset_index(drop=True)
y_train = y_train.iloc[train_order].reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining records:", len(y_train))
print("Testing records:", len(y_test))

print("\nClasses in training:", y_train.nunique())
print("Classes in testing:", y_test.nunique())

print("\nClasses missing from test set:")
print(
    sorted(
        set(label_encoder.transform(singleton_classes))
        - set(y_test.unique())
    )
)

Training features: (80000, 174)
Testing features: (20000, 174)

Training records: 80000
Testing records: 20000

Classes in training: 46
Classes in testing: 42

Classes missing from test set:
[np.int64(27)]


In [30]:
# Validating the split
# Every class should have at least one training record
assert set(y.unique()) == set(
    df.loc[y.index, "disease"].unique()
)

print("Training class distribution:")
print(y_train.map(
    dict(enumerate(label_encoder.classes_))
).value_counts().sort_values())

print("\nTest class distribution:")
print(y_test.map(
    dict(enumerate(label_encoder.classes_))
).value_counts().sort_values())

Training class distribution:
disease_encoded
Metabolic_Syndrome                   1
Urinary_Tract_Infection              2
Acute_Gastroenteritis                2
Psoriasis                            2
Typhoid_Fever                        2
Fanconi_Anemia                       3
Crohn_Disease                        3
Chronic_Liver_Disease                4
Ulcerative_Colitis                   6
Gaucher_Disease                     10
Multiple_Sclerosis                  10
Niemann_Pick_Disease                11
Inflammatory_Bowel_Disease          11
Spinal_Muscular_Atrophy             14
Cirrhosis                           14
Dengue                              14
Ehlers_Danlos_Syndrome              19
Marfan_Syndrome                     23
Osteopetrosis                       37
Amyotrophic_Lateral_Sclerosis       40
Stroke                              51
Tuberculosis                        54
Generalized_Anxiety_Disorder        76
Systemic_Lupus_Erythematosus       125
Cystic_Fibrosis    

In [31]:
from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)

pd.Series(y_train, name="disease_encoded").to_csv(
    processed_dir / "y_train.csv", index=False
)

pd.Series(y_test, name="disease_encoded").to_csv(
    processed_dir / "y_test.csv", index=False
)

pd.DataFrame({
    "class_id": range(len(label_encoder.classes_)),
    "disease": label_encoder.classes_
}).to_csv(
    processed_dir / "label_mapping.csv", index=False
)

print("Preprocessed datasets saved successfully.")

Preprocessed datasets saved successfully.


In [32]:
# Identifying missing disease classes
# Convert encoded labels back to disease names
train_classes = set(y_train.unique())
test_classes = set(y_test.unique())

all_classes = set(range(len(label_encoder.classes_)))

missing_test_ids = sorted(all_classes - test_classes)

print("Classes absent from the test set:")

for class_id in missing_test_ids:
    disease_name = label_encoder.inverse_transform([class_id])[0]
    total_records = int((df["disease"] == disease_name).sum())

    print(
        f"{disease_name}: "
        f"{total_records} record(s), "
        f"class ID = {class_id}"
    )

Classes absent from the test set:
Metabolic_Syndrome: 1 record(s), class ID = 27
Psoriasis: 2 record(s), class ID = 37
Typhoid_Fever: 2 record(s), class ID = 43
Urinary_Tract_Infection: 2 record(s), class ID = 45


In [33]:
# Saving datasets
import joblib
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    label_encoder,
    processed_dir / "label_encoder.joblib"
)

print("Label encoder saved successfully.")

Label encoder saved successfully.
